# Python for Data Engineering — Crash Course> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## 1. Környezet és csomagok

In [ ]:
%pip install pandas pyarrow pydantic requests --quiet

In [ ]:
import sys, platformprint('Python:', sys.version)print('Platform:', platform.platform())

## 2. Idiomatikus Python — list comprehension, generator

In [ ]:
# List comprehension vs hagyományos ciklusnumbers = range(1, 11)# Régi módonsquares_old = []for n in numbers:    if n % 2 == 0:        squares_old.append(n ** 2)# Pythonikus módonsquares_new = [n ** 2 for n in numbers if n % 2 == 0]print('régi:', squares_old)print('új  :', squares_new)# Generator — lusta kiértékelés, nagy adatokhozgen = (n ** 2 for n in numbers if n % 2 == 0)print('gen :', list(gen))

## 3. pathlib — modern fájlkezelés

In [ ]:
from pathlib import Pathdata_dir = Path('data')data_dir.mkdir(exist_ok=True)sample = data_dir / 'sample.txt'sample.write_text('Hello, data engineering!\nÉkezetes karakterek: ÁÉÍÓŐÚŰ')print('Fájl mérete:', sample.stat().st_size, 'bájt')print('Abszolút út:', sample.resolve())print('\nTartalom:\n', sample.read_text())

## 4. pandas DataFrame — ETL alapok

In [ ]:
import pandas as pd# Minta adat: WebShop Pro rendelésekdf = pd.DataFrame({    'order_id':    [101, 102, 103, 104, 105],    'customer_id': [1, 1, 2, 3, 1],    'amount':      [12500, 8900, 24000, 5200, 3400],    'status':      ['paid', 'paid', 'pending', 'paid', 'cancelled']})# Transzformáció: csak fizetett rendelések + aggregációsummary = (    df.query("status == 'paid'")      .groupby('customer_id', as_index=False)      .agg(orders=('order_id', 'count'), revenue=('amount', 'sum'))      .sort_values('revenue', ascending=False))summary

## 5. Parquet I/O — lakehouse-kompatibilis tárolás

In [ ]:
from pathlib import Pathout = Path('data/orders.parquet')df.to_parquet(out, engine='pyarrow', compression='snappy')print('Írt méret:', out.stat().st_size, 'bájt')# Visszaolvasásdf2 = pd.read_parquet(out)print('\nVisszaolvasva:')print(df2)

## 6. Pydantic — runtime séma validáció

In [ ]:
from pydantic import BaseModel, Field, ValidationErrorfrom datetime import datetimeclass Order(BaseModel):    order_id:    int           = Field(ge=1)    customer_id: int           = Field(ge=1)    amount:      float         = Field(gt=0)    status:      str           = Field(pattern='^(paid|pending|cancelled)$')    created_at:  datetime      = Field(default_factory=datetime.now)# Érvényes rekordvalid = Order(order_id=1, customer_id=42, amount=9999.99, status='paid')print('OK:', valid.model_dump_json())# Érvénytelen — elkapjuk a hibáttry:    bad = Order(order_id=0, customer_id=1, amount=-100, status='wrong')except ValidationError as e:    print('\nHibák száma:', len(e.errors()))    for err in e.errors():        print(f"  - {err['loc']}: {err['msg']}")

## 7. Logging — production-ready diagnosztika

In [ ]:
import logging, jsonclass JsonFormatter(logging.Formatter):    def format(self, record):        return json.dumps({            'ts':     self.formatTime(record, '%Y-%m-%dT%H:%M:%S'),            'level':  record.levelname,            'msg':    record.getMessage(),            'logger': record.name,        }, ensure_ascii=False)handler = logging.StreamHandler()handler.setFormatter(JsonFormatter())log = logging.getLogger('etl.orders')log.setLevel(logging.DEBUG)log.handlers = [handler]log.info('Pipeline elindult')log.warning('Lassú forrás: %s', 'api.example.com')log.error('Rekord elutasítva: %s', {'order_id': 0, 'reason': 'invalid amount'})

## 8. Unit test — pytest stílusban

In [ ]:
def transform_order(raw: dict) -> dict:    return {        'order_id':    int(raw['id']),        'customer_id': int(raw['customer']),        'amount':      round(float(raw['amt']), 2),        'status':      raw.get('status', 'unknown').lower(),    }# Tesztek (Jupyterben inline futnak)def test_transform_order_ok():    result = transform_order({'id': '42', 'customer': '1', 'amt': '99.999', 'status': 'PAID'})    assert result == {'order_id': 42, 'customer_id': 1, 'amount': 100.0, 'status': 'paid'}def test_transform_order_default_status():    result = transform_order({'id': '1', 'customer': '1', 'amt': '10'})    assert result['status'] == 'unknown'test_transform_order_ok()test_transform_order_default_status()print('✓ minden teszt sikeres')

## Következő lépések- Térj vissza a [web-alapú kurzushoz](python-data-engineering/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*